In [1]:
import pandas as pd

# نقرأ الملف لجدول (DataFrame)
df = pd.read_csv("cleaned_uci_online_sale_dataset.csv")

print("حجم الداتا:", df.shape)
print("\n--- أول 5 صفوف ---")
print(df.head())

print("\n--- أنواع الأعمدة ---")
print(df.dtypes)

حجم الداتا: (531282, 8)

--- أول 5 صفوف ---
   InvoiceNo StockCode                          Description  Quantity  \
0     536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1     536365     71053                  WHITE METAL LANTERN         6   
2     536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3     536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4     536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  

--- أنواع الأعمدة ---
InvoiceNo        int64
StockCode       object
Description     object
Quantity         int64
InvoiceDate     ob

In [2]:
print("\n--- القيم الناقصة بكل عمود ---")
print(df.isnull().sum())
print("\n--- ملخص إحصائي للأعمدة الرقمية ---")
print(df.describe())


--- القيم الناقصة بكل عمود ---
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

--- ملخص إحصائي للأعمدة الرقمية ---
           InvoiceNo       Quantity      UnitPrice     CustomerID
count  531282.000000  531282.000000  531282.000000  531282.000000
mean   559971.886190      10.655317       3.878140   11455.263062
std     13430.741724     156.830764      32.510663    6795.268735
min    536365.000000       1.000000       0.000000       0.000000
25%    547903.000000       1.000000       1.250000       0.000000
50%    560693.000000       3.000000       2.080000   14375.000000
75%    571846.000000      10.000000       4.130000   16261.000000
max    581587.000000   80995.000000   13541.330000   18287.000000


In [3]:
# نحسب كم صف عندهم CustomerID = 0 بالضبط
zero_customer = (df['CustomerID'] == 0).sum()
print(f"عدد الصفوف بدون عميل معروف: {zero_customer}")
print(f"النسبة المئوية: {zero_customer / len(df) * 100:.2f}%")

# نتحقق من صفوف UnitPrice = 0 (منتجات مجانية/أخطاء)
zero_price = (df['UnitPrice'] == 0).sum()
print(f"\nعدد الصفوف بسعر = 0: {zero_price}")
print(df[df['UnitPrice'] == 0].head(10))

عدد الصفوف بدون عميل معروف: 133358
النسبة المئوية: 25.10%

عدد الصفوف بسعر = 0: 1179
      InvoiceNo StockCode          Description  Quantity          InvoiceDate  \
613      536414     22139  Missing_Description        56  2010-12-01 11:52:00   
1958     536545     21134  Missing_Description         1  2010-12-01 14:32:00   
1959     536546     22145  Missing_Description         1  2010-12-01 14:33:00   
1960     536547     37509  Missing_Description         1  2010-12-01 14:33:00   
1961     536549    85226A  Missing_Description         1  2010-12-01 14:34:00   
1962     536550     85044  Missing_Description         1  2010-12-01 14:34:00   
1998     536552     20950  Missing_Description         1  2010-12-01 14:34:00   
1999     536553     37461  Missing_Description         3  2010-12-01 14:35:00   
2000     536554     84670  Missing_Description        23  2010-12-01 14:35:00   
4309     536765    84952C  Missing_Description        19  2010-12-02 14:43:00   

      UnitPrice  Custom

In [4]:

print("الحجم قبل التنظيف:", df.shape)

# ----------------------------------------------------
# 1) حذف الصفوف بدون عميل معروف (CustomerID = 0)
# سبب منطقي ثابت: هدف المشروع "سلوك عميل حقيقي"، مش معاملات مجهولة
# هاد قرار "شرط منطقي" مش إحصائية متعلّمة من البيانات، فآمن نسويه هلق
df = df[df['CustomerID'] != 0]

# ----------------------------------------------------
# 2) تحويل InvoiceDate من نص (object) لنوع datetime حقيقي
# ضروري لأي عملية حسابية زمنية لاحقًا (RFM بالكامل مبني على فروقات تواريخ)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("الحجم بعد حذف CustomerID=0:", df.shape)
print("\nنوع عمود InvoiceDate الآن:", df['InvoiceDate'].dtype)
print("أقدم تاريخ بالداتا:", df['InvoiceDate'].min())
print("أحدث تاريخ بالداتا:", df['InvoiceDate'].max())

الحجم قبل التنظيف: (531282, 8)
الحجم بعد حذف CustomerID=0: (397924, 8)

نوع عمود InvoiceDate الآن: datetime64[ns]
أقدم تاريخ بالداتا: 2010-12-01 08:26:00
أحدث تاريخ بالداتا: 2011-12-09 12:50:00


In [5]:
# نشوف الصفوف يلي سعرها صفر بالضبط
zero_price = df[df['UnitPrice'] == 0]
print(f"عدد الصفوف بسعر = 0: {len(zero_price)}")
print(f"النسبة: {len(zero_price)/len(df)*100:.3f}%")
print("\n--- عينة من هاي الصفوف ---")
print(zero_price[['Description', 'Quantity', 'UnitPrice']].head(15))

# نتأكد كمان: هل لسه في كميات سالبة أو صفرية بعد التنظيف السابق؟
print(f"\nأصغر قيمة Quantity: {df['Quantity'].min()}")
print(f"عدد الصفوف بكمية <= 0: {(df['Quantity'] <= 0).sum()}")

عدد الصفوف بسعر = 0: 40
النسبة: 0.010%

--- عينة من هاي الصفوف ---
                                Description  Quantity  UnitPrice
9174           ROUND CAKE TIN VINTAGE GREEN         1        0.0
32939          ADVENT CALENDAR GINGHAM SACK         4        0.0
39317              REGENCY CAKESTAND 3 TIER        10        0.0
46139               PAPER BUNTING RETROSPOT        24        0.0
46141                PLASTERS IN TIN SKULLS        24        0.0
55562         ORGANISER WOOD ANTIQUE WHITE          1        0.0
85044          FAIRY CAKES NOTEBOOK A6 SIZE        16        0.0
127556  CERAMIC BOWL WITH LOVE HEART DESIGN        36        0.0
136647   MINI CAKE STAND  HANGING STRAWBERY         5        0.0
142177          HEART GARLAND RUSTIC PADDED         2        0.0
153627   CHILDS BREAKFAST SET CIRCUS PARADE         1        0.0
183632                        PARTY BUNTING         4        0.0
194186            SET OF 6 SOLDIER SKITTLES        80        0.0
273382           OVAL W

In [6]:
# نحذف الصفوف بسعر = 0 (تأثير مهمل، ما بيضيف قيمة لتحليل Monetary)
df = df[df['UnitPrice'] > 0]

print("الحجم بعد حذف UnitPrice=0:", df.shape)

# ----------------------------------------------------
# نضيف عمود جديد: TotalPrice = القيمة الفعلية لكل سطر فاتورة (Quantity × UnitPrice)
# هاد العمود أساسي لحساب Monetary لاحقًا (قديش العميل صرف مجموع)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print("\n--- أول 5 صفوف بعد إضافة TotalPrice ---")
print(df[['InvoiceNo', 'CustomerID', 'InvoiceDate', 'Quantity', 'UnitPrice', 'TotalPrice']].head())

الحجم بعد حذف UnitPrice=0: (397884, 8)

--- أول 5 صفوف بعد إضافة TotalPrice ---
   InvoiceNo  CustomerID         InvoiceDate  Quantity  UnitPrice  TotalPrice
0     536365     17850.0 2010-12-01 08:26:00         6       2.55       15.30
1     536365     17850.0 2010-12-01 08:26:00         6       3.39       20.34
2     536365     17850.0 2010-12-01 08:26:00         8       2.75       22.00
3     536365     17850.0 2010-12-01 08:26:00         6       3.39       20.34
4     536365     17850.0 2010-12-01 08:26:00         6       3.39       20.34


In [7]:

# نحدد نقطة القطع: 3 شهور قبل آخر تاريخ بالداتا
cutoff_date = df['InvoiceDate'].max() - pd.DateOffset(months=3)
print("نقطة القطع الزمني:", cutoff_date)

# ----------------------------------------------------
# نقسم الداتا لقسمين بناءً على التاريخ (مو train_test_split عشوائي!)
observation_df = df[df['InvoiceDate'] <= cutoff_date]   # فترة الملاحظة - لبناء RFM
prediction_df = df[df['InvoiceDate'] > cutoff_date]      # فترة التنبؤ - لبناء Target

print("\nحجم فترة الملاحظة:", observation_df.shape)
print("حجم فترة التنبؤ:", prediction_df.shape)

print("\nعدد العملاء الفريدين بفترة الملاحظة:", observation_df['CustomerID'].nunique())
print("عدد العملاء الفريدين بفترة التنبؤ:", prediction_df['CustomerID'].nunique())

نقطة القطع الزمني: 2011-09-09 12:50:00

حجم فترة الملاحظة: (235682, 9)
حجم فترة التنبؤ: (162202, 9)

عدد العملاء الفريدين بفترة الملاحظة: 3365
عدد العملاء الفريدين بفترة التنبؤ: 2893


In [8]:

# نحدد "نقطة اليوم" (Snapshot Date) = نهاية فترة الملاحظة بالضبط
# منها منحسب Recency (كل الحسابات الزمنية لازم تكون بالنسبة لنفس النقطة المرجعية)
snapshot_date = cutoff_date + pd.Timedelta(days=1)

# ----------------------------------------------------
# .groupby('CustomerID') بيجمع كل صفوف نفس العميل سوا
# .agg({...}) بيحدد "شو نسوي" لكل عمود بعد التجميع
rfm = observation_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,   # Recency: أيام من آخر شراء لحد نقطة اللقطة
    'InvoiceNo': 'nunique',                                     # Frequency: عدد الفواتير الفريدة (مو عدد الصفوف!)
    'TotalPrice': 'sum'                                         # Monetary: مجموع كل المبالغ المصروفة
})

# نعيد تسمية الأعمدة لأسماء واضحة
rfm.columns = ['Recency', 'Frequency', 'Monetary']

print("حجم جدول RFM:", rfm.shape)
print("\n--- أول 10 عملاء ---")
print(rfm.head(10))

print("\n--- ملخص إحصائي ---")
print(rfm.describe())

حجم جدول RFM: (3365, 3)

--- أول 10 عملاء ---
            Recency  Frequency  Monetary
CustomerID                              
12346.0         235          1  77183.60
12347.0          39          5   2790.86
12348.0         158          3   1487.24
12350.0         219          1    334.40
12352.0         171          5   1561.81
12353.0         113          1     89.00
12354.0         141          1   1079.40
12355.0         123          1    459.40
12356.0         155          2   2753.08
12358.0          60          1    484.86

--- ملخص إحصائي ---
           Recency    Frequency       Monetary
count  3365.000000  3365.000000    3365.000000
mean     94.960178     3.529866    1616.969944
std      79.863689     5.780679    6153.373605
min       1.000000     1.000000       2.900000
25%      26.000000     1.000000     265.850000
50%      74.000000     2.000000     566.380000
75%     151.000000     4.000000    1392.840000
max     283.000000   131.000000  178302.620000


In [9]:
import numpy as np
rfm_log = rfm.copy()
rfm_log['Recency'] = np.log1p(rfm['Recency'])
rfm_log['Frequency'] = np.log1p(rfm['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm['Monetary'])

print("--- قبل التحويل ---")
print(rfm.describe())

print("\n--- بعد تحويل log1p ---")
print(rfm_log.describe())

--- قبل التحويل ---
           Recency    Frequency       Monetary
count  3365.000000  3365.000000    3365.000000
mean     94.960178     3.529866    1616.969944
std      79.863689     5.780679    6153.373605
min       1.000000     1.000000       2.900000
25%      26.000000     1.000000     265.850000
50%      74.000000     2.000000     566.380000
75%     151.000000     4.000000    1392.840000
max     283.000000   131.000000  178302.620000

--- بعد تحويل log1p ---
           Recency    Frequency     Monetary
count  3365.000000  3365.000000  3365.000000
mean      4.042830     1.240128     6.419375
std       1.213123     0.627902     1.222840
min       0.693147     0.693147     1.360977
25%       3.295837     0.693147     5.586687
50%       4.317488     1.098612     6.341029
75%       5.023881     1.609438     7.239818
max       5.648974     4.882802    12.091243


In [10]:
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------
# StandardScaler بيحول كل عمود بحيث: المتوسط = 0، الانحراف المعياري = 1
# هيك كل الأبعاد التلاتة تصير "بمقياس موحّد" وما حدا منهم يسيطر على المسافة
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# نحولها رجوع لـ DataFrame (عشان تضل بأسماء أعمدة واضحة)
rfm_scaled = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'], index=rfm_log.index)

print(rfm_scaled.describe())

            Recency     Frequency      Monetary
count  3.365000e+03  3.365000e+03  3.365000e+03
mean  -2.533882e-17 -1.794833e-17  3.716360e-16
std    1.000149e+00  1.000149e+00  1.000149e+00
min   -2.761616e+00 -8.712544e-01 -4.137212e+00
25%   -6.158517e-01 -8.712544e-01 -6.810469e-01
50%    2.264397e-01 -2.254124e-01 -6.407782e-02
75%    8.088188e-01  5.882524e-01  6.710321e-01
max    1.324172e+00  5.802206e+00  4.638963e+00


In [11]:
from sklearn.cluster import KMeans
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------
# نجرب قيم K مختلفة من 2 لغاية 10، ونسجل "الخطأ" (Inertia) لكل وحدة
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia_values.append(kmeans.inertia_)   # inertia = مجموع مربعات المسافات لكل نقطة عن مركز مجموعتها

# ----------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.savefig('elbow_plot.png', dpi=110)
print("تم حفظ الرسمة")

for k, inertia in zip(k_range, inertia_values):
    print(f"K={k}: Inertia={inertia:.2f}")

تم حفظ الرسمة
K=2: Inertia=5067.28
K=3: Inertia=3799.83
K=4: Inertia=3002.38
K=5: Inertia=2561.13
K=6: Inertia=2251.93
K=7: Inertia=2018.62
K=8: Inertia=1858.65
K=9: Inertia=1707.53
K=10: Inertia=1584.36


In [12]:
# نطبق K-Means النهائي بـ K=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

# نشوف كم عميل بكل مجموعة
print("عدد العملاء بكل مجموعة:")
print(rfm['Cluster'].value_counts())

# الأهم: نشوف متوسط R, F, M لكل مجموعة - عشان "نفهم" شو تمثل كل مجموعة فعليًا
print("\n--- متوسط RFM لكل مجموعة (بالقيم الأصلية، مو بعد Log/Scaling) ---")
print(rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2))

عدد العملاء بكل مجموعة:
Cluster
2    1433
0     978
3     484
1     470
Name: count, dtype: int64

--- متوسط RFM لكل مجموعة (بالقيم الأصلية، مو بعد Log/Scaling) ---
         Recency  Frequency  Monetary
Cluster                              
0          82.66       3.42   1534.67
1          16.43      12.19   6842.52
2         156.00       1.20    287.68
3          15.35       2.23    644.57


In [13]:
# ----------------------------------------------------
# نجيب قائمة كل CustomerID الفريدين يلي اشتروا بفترة التنبؤ
active_customers = prediction_df['CustomerID'].unique()

print(f"عدد العملاء يلي اشتروا بفترة التنبؤ: {len(active_customers)}")

# ----------------------------------------------------
# نبني عمود Churn بجدول RFM (يلي فيه عملاء فترة الملاحظة بس)
# الشرط: هل CustomerID موجود بقائمة active_customers؟
# .isin() بترجع True/False لكل صف حسب وجوده بالقائمة
# ~ (النفي) لأنه: إذا "موجود بالنشيطين" = Churn هو False (0)، وإذا "مش موجود" = Churn هو True (1)
rfm['Churn'] = (~rfm.index.isin(active_customers)).astype(int)

print("\nتوزيع Churn:")
print(rfm['Churn'].value_counts())
print("\nنسبة Churn:", (rfm['Churn'].value_counts(normalize=True) * 100).round(2))

عدد العملاء يلي اشتروا بفترة التنبؤ: 2893

توزيع Churn:
Churn
0    1920
1    1445
Name: count, dtype: int64

نسبة Churn: Churn
0    57.06
1    42.94
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

X = rfm[['Recency', 'Frequency', 'Monetary', 'Cluster']]
y = rfm['Churn']

print("شكل X:", X.shape)
print("شكل y:", y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nتوزيع y_train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nتوزيع y_test:")
print(y_test.value_counts(normalize=True).round(3))


شكل X: (3365, 4)
شكل y: (3365,)

توزيع y_train:
Churn
0    0.571
1    0.429
Name: proportion, dtype: float64

توزيع y_test:
Churn
0    0.571
1    0.429
Name: proportion, dtype: float64


In [15]:
# ----------------------------------------------------
# Pipeline بسيطة: Scaling (لأنه Logistic Regression حساس لمقاييس الأعمدة) + الموديل
pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_lr.fit(X_train, y_train)
y_pred = pipeline_lr.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n--- تقرير التصنيف ---")
print(classification_report(y_test, y_pred))

Accuracy: 0.6731

--- تقرير التصنيف ---
              precision    recall  f1-score   support

           0       0.72      0.69      0.71       384
           1       0.61      0.65      0.63       289

    accuracy                           0.67       673
   macro avg       0.67      0.67      0.67       673
weighted avg       0.68      0.67      0.67       673



In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score

# ----------------------------------------------------
# قاموس فيه كل الموديلات يلي بدنا نجربها
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

# ----------------------------------------------------
# نلف على كل موديل، نبني Pipeline له، ندربه، ونسجل نتائجه
for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_churned = f1_score(y_test, y_pred, pos_label=1)   # F1 لفئة Churned تحديدًا (الأهم إلنا)
    
    results.append({"Model": name, "Accuracy": acc, "F1 (Churned)": f1_churned})
    
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))

# ----------------------------------------------------
# جدول مقارنة نهائي مرتب
results_df = pd.DataFrame(results).sort_values("F1 (Churned)", ascending=False)
print("\n\n--- جدول المقارنة النهائي (مرتب حسب F1 لفئة Churned) ---")
print(results_df.to_string(index=False))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.72      0.69      0.71       384
           1       0.61      0.65      0.63       289

    accuracy                           0.67       673
   macro avg       0.67      0.67      0.67       673
weighted avg       0.68      0.67      0.67       673


=== Decision Tree ===
              precision    recall  f1-score   support

           0       0.72      0.69      0.71       384
           1       0.61      0.64      0.62       289

    accuracy                           0.67       673
   macro avg       0.66      0.67      0.67       673
weighted avg       0.67      0.67      0.67       673


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.68      0.67      0.67       384
           1       0.57      0.57      0.57       289

    accuracy                           0.63       673
   macro avg       0.62      0.62      0.62       673

In [19]:
# ----------------------------------------------------
# AvgOrderValue = Monetary (إجمالي الصرف) ÷ Frequency (عدد الفواتير)
# بيعطينا: "بالمتوسط، قديش العميل يصرف بكل زيارة شراء وحدة؟"
rfm['AvgOrderValue'] = rfm['Monetary'] / rfm['Frequency']

print(rfm[['Recency', 'Frequency', 'Monetary', 'AvgOrderValue']].head(10))

            Recency  Frequency  Monetary  AvgOrderValue
CustomerID                                             
12346.0         235          1  77183.60   77183.600000
12347.0          39          5   2790.86     558.172000
12348.0         158          3   1487.24     495.746667
12350.0         219          1    334.40     334.400000
12352.0         171          5   1561.81     312.362000
12353.0         113          1     89.00      89.000000
12354.0         141          1   1079.40    1079.400000
12355.0         123          1    459.40     459.400000
12356.0         155          2   2753.08    1376.540000
12358.0          60          1    484.86     484.860000


In [20]:
# ----------------------------------------------------
# بعض العملاء ممكن يكون عندهم أكتر من قيمة Country بالبيانات (نادر بس ممكن، خطأ إدخال)
# فمنستخدم .mode()[0] يعني "القيمة الأكتر تكرارًا" لكل عميل، أو ببساطة أول قيمة لو كلهم متطابقين
customer_country = observation_df.groupby('CustomerID')['Country'].agg(lambda x: x.mode()[0])

# ندمجها بجدول RFM (merge بناءً على CustomerID المشترك بين الجدولين)
rfm = rfm.merge(customer_country.rename('Country'), left_index=True, right_index=True)

print(rfm['Country'].value_counts().head(10))

Country
United Kingdom     3030
Germany              74
France               70
Belgium              23
Spain                20
Switzerland          16
Portugal             13
Channel Islands       9
Italy                 9
Australia             9
Name: count, dtype: int64


In [21]:
# ----------------------------------------------------
# نبسّط عمود Country لفئتين بس: هل العميل من UK ولا لأ؟
# القرار مبني على النظر لتوزيع value_counts() - هيمنة UK الساحقة تبرر هالتبسيط
rfm['IsUK'] = (rfm['Country'] == 'United Kingdom').astype(int)

print(rfm['IsUK'].value_counts())
print("\nنسبة UK:", (rfm['IsUK'].value_counts(normalize=True) * 100).round(2))

# نشيل عمود Country النصي الأصلي - عوضناه بـ IsUK الرقمي
rfm = rfm.drop('Country', axis=1)

IsUK
1    3030
0     335
Name: count, dtype: int64

نسبة UK: IsUK
1    90.04
0     9.96
Name: proportion, dtype: float64


In [22]:

from sklearn.metrics import classification_report, accuracy_score, f1_score

# ----------------------------------------------------
# X الجديد يتضمن الـ Features الأصلية + الاثنين الجداد
X = rfm[['Recency', 'Frequency', 'Monetary', 'Cluster', 'AvgOrderValue', 'IsUK']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    # ----------------------------------------------------
    # نتنبأ على الاثنين: Train و Test - عشان نقارن ونكتشف Overfitting
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, pos_label=1)
    
    # الفجوة بين أداء Train و Test - المؤشر المباشر على Overfitting
    gap = train_acc - test_acc
    
    results.append({
        "Model": name,
        "Train Acc": round(train_acc, 4),
        "Test Acc": round(test_acc, 4),
        "Gap": round(gap, 4),
        "Test F1 (Churned)": round(test_f1, 4)
    })

results_df = pd.DataFrame(results).sort_values("Test F1 (Churned)", ascending=False)
print(results_df.to_string(index=False))

              Model  Train Acc  Test Acc    Gap  Test F1 (Churned)
Logistic Regression     0.6761    0.6731 0.0030             0.6309
      Decision Tree     0.7147    0.6523 0.0624             0.6176
      Random Forest     1.0000    0.6330 0.3670             0.5835
                KNN     0.7608    0.6226 0.1382             0.5544


In [23]:
from sklearn.model_selection import GridSearchCV

# ----------------------------------------------------
# لكل موديل، نحدد شبكة بارامترات تركز على "تبسيط" الموديل (منع Overfitting)
param_grids = {
    "Decision Tree": {
        "model__max_depth": [2, 3, 4, 5, 6],              # عمق أقل = شجرة أبسط = أقل حفظ
        "model__min_samples_leaf": [5, 10, 20, 30]         # أقل عدد عينات بكل ورقة - يمنع تفرعات دقيقة جدًا لعينة واحدة
    },
    "Random Forest": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [3, 5, 7, 10],                  # هون الأهم - نحد العمق (كان بدون حد = سبب الـOverfitting)
        "model__min_samples_leaf": [5, 10, 20]
    },
    "KNN": {
        "model__n_neighbors": [5, 10, 15, 20, 25]           # جيران أكتر = قرار أقل حساسية لعميل واحد شاذ = أعمم
    },
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]                 # C أصغر = تنظيم (regularization) أقوى = موديل أبسط
    }
}

model_objects = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000)
}

best_models = {}
tuning_results = []

for name in models.keys():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model_objects[name])
    ])
    
    grid = GridSearchCV(
        pipeline,
        param_grids[name],
        cv=5,                              # 5-Fold Cross-Validation - تقييم أوثق من تقسيم واحد بس
        scoring='f1',                      # نحسّن على أساس F1 (مش Accuracy) - لأنه أهم إلنا لفئة Churned
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    
    best_models[name] = grid.best_estimator_
    
    y_train_pred = grid.best_estimator_.predict(X_train)
    y_test_pred = grid.best_estimator_.predict(X_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, pos_label=1)
    
    tuning_results.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "Train Acc": round(train_acc, 4),
        "Test Acc": round(test_acc, 4),
        "Gap": round(train_acc - test_acc, 4),
        "Test F1 (Churned)": round(test_f1, 4)
    })

tuning_df = pd.DataFrame(tuning_results).sort_values("Test F1 (Churned)", ascending=False)
print(tuning_df.to_string(index=False))

              Model                                                                       Best Params  Train Acc  Test Acc    Gap  Test F1 (Churned)
      Decision Tree                             {'model__max_depth': 4, 'model__min_samples_leaf': 5}     0.6965    0.6701 0.0264             0.6531
                KNN                                                        {'model__n_neighbors': 25}     0.6902    0.6850 0.0052             0.6319
Logistic Regression                                                                  {'model__C': 10}     0.6768    0.6716 0.0052             0.6298
      Random Forest {'model__max_depth': 3, 'model__min_samples_leaf': 10, 'model__n_estimators': 50}     0.6839    0.6716 0.0123             0.6298
